# 🔊 Detecting Centrifugal-Pump Cavitation Using Sound
## Mechanical Engineering + Signal Processing + Deep Learning

This notebook turns a short pump recording into a log Mel-spectrogram and uses a 2D CNN to classify **Normal**, **Mild Cavitation**, or **Severe Cavitation**.

> All operating responses are simulated educational recommendations. They are not instructions for real industrial equipment.

👉 **Open the interactive companion:** [https://pump-cavitation-audio.streamlit.app](https://pump-cavitation-audio.streamlit.app/?stage=start)

## The complete system

Pump sound → normalized waveform → Mel-spectrogram → 2D CNN → condition probability → simulated engineering response.

Pump RPM, inlet pressure, flow rate, and valve opening are retained as engineering metadata for generation and labelling. **Only the spectrogram enters the CNN.**

## Interactive learning journey

- [What Happens Inside the Pump](https://pump-cavitation-audio.streamlit.app/?stage=cavitation) — The Detection Target
- [A Short Pump Recording](https://pump-cavitation-audio.streamlit.app/?stage=listen) — Audio Input
- [Sound as Pressure Over Time](https://pump-cavitation-audio.streamlit.app/?stage=waveform) — Waveform Preprocessing
- [Making Cavitation Visible](https://pump-cavitation-audio.streamlit.app/?stage=spectrogram) — Mel-Spectrogram
- [An Automated Acoustic Inspector](https://pump-cavitation-audio.streamlit.app/?stage=cnn) — 2D CNN
- [Learning From Labelled Pump Tests](https://pump-cavitation-audio.streamlit.app/?stage=training) — Supervised Training
- [From Sound to Pump Condition](https://pump-cavitation-audio.streamlit.app/?stage=prediction) — Softmax Prediction
- [The Mechanical Engineering Audit](https://pump-cavitation-audio.streamlit.app/?stage=audit) — Confusion Matrix and Limitations

In [ ]:
# Colab setup: uncomment if packages are missing.
# !pip -q install librosa tensorflow scikit-learn soundfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa, librosa.display
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
SEED=42; np.random.seed(SEED); tf.random.set_seed(SEED)
SR,DURATION,N_MELS=8000,4,64
CLASSES=["Normal","Mild Cavitation","Severe Cavitation"]
print("Audio duration:",DURATION,"s | sample rate:",SR,"Hz")

---
# 1. What Happens Inside the Pump
### Phase 1 of 6 · Inside the Pump

## Part 1 · At the pump
At low local pressure, liquid vaporises into bubbles near the impeller eye. Those bubbles move into higher-pressure regions and collapse violently.

## Part 2 · Engineering challenge
Cavitation can erode the impeller, reduce efficiency, increase vibration, and shorten pump life before damage is visible externally.

## Part 3 · Where AI comes in
The collapsing bubbles produce distinctive broadband and impulsive sound. That acoustic signature becomes the observable target.

**Mechanical Engineering:** What Happens Inside the Pump → **AI:** The Detection Target → `bubble formation and collapse`

> 🎬 **See this illustrated and interactive:** [https://pump-cavitation-audio.streamlit.app/?stage=cavitation](https://pump-cavitation-audio.streamlit.app/?stage=cavitation)

## Part 4 · Technical explanation

Cavitation begins when local pressure falls below the liquid's vapour pressure. Bubbles form near the impeller eye and collapse after entering a higher-pressure region. The collapse produces pressure pulses, broadband noise, vibration, and potentially erosion.

The notebook does not claim that sound alone proves a hydraulic diagnosis. It demonstrates how sound can contribute to condition monitoring.

## Part 5 · What you just built

**In the notebook:** Generate labelled normal, mild, and severe pump recordings.

**Takeaway:** Cavitation is a hydraulic phenomenon that leaves an acoustic signature.

[Project overview](https://pump-cavitation-audio.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: A Short Pump Recording](https://pump-cavitation-audio.streamlit.app/?stage=listen) ▶

---
# 2. A Short Pump Recording
### Phase 2 of 6 · Listening to the Machine

## Part 1 · At the pump
A microphone records several seconds of pump operation while RPM, inlet pressure, flow, and valve opening describe the test condition.

## Part 2 · Engineering challenge
A person can hear obvious gravel-like noise, but continuous listening is impractical and early changes can be masked by motors, bearings, and background noise.

## Part 3 · Where AI comes in
Use only the recording as the CNN input. Engineering variables help generate or label examples but do not enter the image model.

**Mechanical Engineering:** A Short Pump Recording → **AI:** Audio Input → `2-5 second mono waveform`

> 🎬 **See this illustrated and interactive:** [https://pump-cavitation-audio.streamlit.app/?stage=listen](https://pump-cavitation-audio.streamlit.app/?stage=listen)

## Part 4 · Technical explanation

In [ ]:
def synth_clip(label,seed):
    rng=np.random.default_rng(seed); t=np.arange(SR*DURATION)/SR
    rpm=rng.uniform(1100,2200); shaft=rpm/60
    x=.34*np.sin(2*np.pi*shaft*t)+.17*np.sin(2*np.pi*2*shaft*t)+.08*np.sin(2*np.pi*3*shaft*t)
    x+=.025*rng.normal(size=len(t))
    severity=[0,.45,1.0][label]
    impulses=rng.random(len(t)) < severity*.006
    crack=np.convolve(impulses*rng.normal(size=len(t)),np.exp(-np.arange(70)/13),mode="same")
    x+=severity*(.18*rng.normal(size=len(t))+.8*crack)
    inlet=max(.2,1.8-1.15*severity+rng.normal(0,.08)); flow=80-18*severity+rng.normal(0,3)
    valve=92-30*severity+rng.normal(0,3)
    return x.astype("float32"),dict(rpm=rpm,inlet_pressure_bar=inlet,flow_l_min=flow,valve_open_pct=valve)

x,meta=synth_clip(2,7)
print(meta); print("waveform samples:",x.shape)
plt.figure(figsize=(13,3));plt.plot(np.arange(len(x))/SR,x,linewidth=.5);plt.xlabel("Time (s)");plt.ylabel("Amplitude");plt.grid(alpha=.2);plt.show()

## Part 5 · What you just built

**In the notebook:** Create four-second 8 kHz audio clips and store engineering metadata separately.

**Takeaway:** The model listens to the sound; metadata explains how the sound was produced.

◀ [Previous: What Happens Inside the Pump](https://pump-cavitation-audio.streamlit.app/?stage=cavitation) &nbsp;|&nbsp; [Project overview](https://pump-cavitation-audio.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Sound as Pressure Over Time](https://pump-cavitation-audio.streamlit.app/?stage=waveform) ▶

---
# 3. Sound as Pressure Over Time
### Phase 3 of 6 · Making Sound Visible

## Part 1 · At the pump
The microphone converts pressure fluctuations into a sequence of amplitude samples.

## Part 2 · Engineering challenge
Raw loudness varies with microphone distance and gain, so volume alone must not become the classifier.

## Part 3 · Where AI comes in
Remove DC offset and normalize every clip to a common peak before extracting its frequency content.

**Mechanical Engineering:** Sound as Pressure Over Time → **AI:** Waveform Preprocessing → `remove mean; normalize amplitude`

> 🎬 **See this illustrated and interactive:** [https://pump-cavitation-audio.streamlit.app/?stage=waveform](https://pump-cavitation-audio.streamlit.app/?stage=waveform)

## Part 4 · Technical explanation

In [ ]:
def normalize_audio(x):
    x=x-np.mean(x)
    return x/(np.max(np.abs(x))+1e-9)

x_norm=normalize_audio(x)
print("Mean:",x_norm.mean().round(6),"Peak:",np.abs(x_norm).max())

## Part 5 · What you just built

**In the notebook:** Apply mean removal and peak normalization to every waveform.

**Takeaway:** Normalize level so the CNN learns sound structure rather than microphone gain.

◀ [Previous: A Short Pump Recording](https://pump-cavitation-audio.streamlit.app/?stage=listen) &nbsp;|&nbsp; [Project overview](https://pump-cavitation-audio.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Making Cavitation Visible](https://pump-cavitation-audio.streamlit.app/?stage=spectrogram) ▶

---
# 4. Making Cavitation Visible
### Phase 3 of 6 · Making Sound Visible

## Part 1 · At the pump
Normal rotation creates stable tonal bands. Bubble collapse adds scattered high-frequency energy and short bursts that change over time.

## Part 2 · Engineering challenge
A waveform is too dense to interpret directly; frequency and timing are mixed into one line.

## Part 3 · Where AI comes in
A Mel-spectrogram separates frequency vertically and time horizontally, turning acoustic behaviour into a 2D image.

**Mechanical Engineering:** Making Cavitation Visible → **AI:** Mel-Spectrogram → `STFT -> Mel bands -> log power`

> 🎬 **See this illustrated and interactive:** [https://pump-cavitation-audio.streamlit.app/?stage=spectrogram](https://pump-cavitation-audio.streamlit.app/?stage=spectrogram)

## Part 4 · Technical explanation

In [ ]:
def mel_image(x):
    x=normalize_audio(x)
    mel=librosa.feature.melspectrogram(y=x,sr=SR,n_fft=512,hop_length=128,n_mels=N_MELS,fmin=20,fmax=SR/2)
    return librosa.power_to_db(mel,ref=np.max).astype("float32")

fig,ax=plt.subplots(1,3,figsize=(16,4))
for label,name in enumerate(CLASSES):
    sample,_=synth_clip(label,100+label)
    librosa.display.specshow(mel_image(sample),sr=SR,hop_length=128,x_axis="time",y_axis="mel",ax=ax[label],cmap="magma")
    ax[label].set_title(name)
plt.tight_layout();plt.show()

## Part 5 · What you just built

**In the notebook:** Compute a 64-band log Mel-spectrogram for every clip.

**Takeaway:** The spectrogram reveals where acoustic energy occurs in frequency and time.

◀ [Previous: Sound as Pressure Over Time](https://pump-cavitation-audio.streamlit.app/?stage=waveform) &nbsp;|&nbsp; [Project overview](https://pump-cavitation-audio.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: An Automated Acoustic Inspector](https://pump-cavitation-audio.streamlit.app/?stage=cnn) ▶

---
# 5. An Automated Acoustic Inspector
### Phase 4 of 6 · Training the Inspector

## Part 1 · At the pump
An engineer compares stable harmonic bands with broadband crackle and repeated impulsive events.

## Part 2 · Engineering challenge
Hand-written frequency thresholds fail when RPM, load, microphone, and background noise change.

## Part 3 · Where AI comes in
Convolutional filters learn local time-frequency shapes and combine them into condition evidence without manually specifying every pattern.

**Mechanical Engineering:** An Automated Acoustic Inspector → **AI:** 2D CNN → `Conv2D -> pool -> Conv2D -> pool -> dense`

> 🎬 **See this illustrated and interactive:** [https://pump-cavitation-audio.streamlit.app/?stage=cnn](https://pump-cavitation-audio.streamlit.app/?stage=cnn)

## Part 4 · Technical explanation

In [ ]:
X,y,metadata=[],[],[]
for label in range(3):
    for run in range(120):
        audio,meta=synth_clip(label,10000*label+run)
        X.append(mel_image(audio));y.append(label);metadata.append(meta)
X=np.array(X)[...,None];y=np.array(y)
print("CNN input:",X.shape,"labels:",y.shape)

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,stratify=y,random_state=SEED)
X_train,X_val,y_train,y_val=train_test_split(X_train,y_train,test_size=.20,stratify=y_train,random_state=SEED)
model=Sequential([Input(shape=X.shape[1:]),Conv2D(16,3,activation="relu"),MaxPooling2D(),
                  Conv2D(32,3,activation="relu"),MaxPooling2D(),Flatten(),Dense(48,activation="relu"),
                  Dropout(.25),Dense(3,activation="softmax")])
model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])
model.summary()

## Part 5 · What you just built

**In the notebook:** Build a compact 2D CNN ending in three Softmax probabilities.

**Takeaway:** A CNN treats the spectrogram as an image of machine behaviour.

◀ [Previous: Making Cavitation Visible](https://pump-cavitation-audio.streamlit.app/?stage=spectrogram) &nbsp;|&nbsp; [Project overview](https://pump-cavitation-audio.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Learning From Labelled Pump Tests](https://pump-cavitation-audio.streamlit.app/?stage=training) ▶

---
# 6. Learning From Labelled Pump Tests
### Phase 4 of 6 · Training the Inspector

## Part 1 · At the pump
Controlled tests provide examples labelled Normal, Mild Cavitation, and Severe Cavitation.

## Part 2 · Engineering challenge
Randomly splitting near-duplicate clips can make accuracy look better than performance on a genuinely new pump run.

## Part 3 · Where AI comes in
Train on earlier generated runs, validate separately, and reserve unseen clips for the final audit.

**Mechanical Engineering:** Learning From Labelled Pump Tests → **AI:** Supervised Training → `cross-entropy + Adam + early stopping`

> 🎬 **See this illustrated and interactive:** [https://pump-cavitation-audio.streamlit.app/?stage=training](https://pump-cavitation-audio.streamlit.app/?stage=training)

## Part 4 · Technical explanation

In [ ]:
early=EarlyStopping(monitor="val_loss",patience=5,restore_best_weights=True)
history=model.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=35,batch_size=24,callbacks=[early],verbose=0)
fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].plot(history.history["loss"],label="train");ax[0].plot(history.history["val_loss"],label="validation");ax[0].set_title("Loss")
ax[1].plot(history.history["accuracy"],label="train");ax[1].plot(history.history["val_accuracy"],label="validation");ax[1].set_title("Accuracy")
for a in ax:a.set_xlabel("Epoch");a.grid(alpha=.2);a.legend()
plt.show()

## Part 5 · What you just built

**In the notebook:** Train with sparse categorical cross-entropy and early stopping.

**Takeaway:** A trustworthy test set must represent recordings the model did not learn by heart.

◀ [Previous: An Automated Acoustic Inspector](https://pump-cavitation-audio.streamlit.app/?stage=cnn) &nbsp;|&nbsp; [Project overview](https://pump-cavitation-audio.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: From Sound to Pump Condition](https://pump-cavitation-audio.streamlit.app/?stage=prediction) ▶

---
# 7. From Sound to Pump Condition
### Phase 5 of 6 · Engineering Decision

## Part 1 · At the pump
The maintenance team needs one condition assessment and evidence strong enough to decide whether to inspect.

## Part 2 · Engineering challenge
A class name alone hides uncertainty and can encourage false confidence in noisy or unfamiliar conditions.

## Part 3 · Where AI comes in
Report all three probabilities, highlight the largest, and attach a simulated educational response.

**Mechanical Engineering:** From Sound to Pump Condition → **AI:** Softmax Prediction → `Normal / Mild / Severe + confidence`

> 🎬 **See this illustrated and interactive:** [https://pump-cavitation-audio.streamlit.app/?stage=prediction](https://pump-cavitation-audio.streamlit.app/?stage=prediction)

## Part 4 · Technical explanation

In [ ]:
i=5;probs=model.predict(X_test[i:i+1],verbose=0)[0];pred=int(np.argmax(probs))
print("Pump Condition:",CLASSES[pred]);print("Confidence:",f"{probs[pred]:.1%}")
for name,p in zip(CLASSES,probs):print(f"  {name:18s} {p:.1%}")
responses={0:"Continue simulated operation",1:"Reduce simulated speed or inspect inlet condition",2:"Stop or reduce simulated operation and inspect the pump system"}
print("Recommended simulated response:",responses[pred])
plt.figure(figsize=(10,4));librosa.display.specshow(X_test[i,:,:,0],sr=SR,hop_length=128,x_axis="time",y_axis="mel",cmap="magma");plt.colorbar(format="%+2.0f dB");plt.title(CLASSES[pred]);plt.show()

## Part 5 · What you just built

**In the notebook:** Classify a test recording and display confidence with its spectrogram.

**Takeaway:** A prediction supports inspection; it does not operate industrial equipment.

◀ [Previous: Learning From Labelled Pump Tests](https://pump-cavitation-audio.streamlit.app/?stage=training) &nbsp;|&nbsp; [Project overview](https://pump-cavitation-audio.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: The Mechanical Engineering Audit](https://pump-cavitation-audio.streamlit.app/?stage=audit) ▶

---
# 8. The Mechanical Engineering Audit
### Phase 6 of 6 · Model Audit

## Part 1 · At the pump
Missing severe cavitation is more consequential than confusing mild cavitation with severe cavitation.

## Part 2 · Engineering challenge
Overall accuracy can hide a model that performs badly on the most important class or on a different pump and microphone.

## Part 3 · Where AI comes in
Inspect the confusion matrix and per-class recall, then state domain-shift and synthetic-data limitations explicitly.

**Mechanical Engineering:** The Mechanical Engineering Audit → **AI:** Confusion Matrix and Limitations → `accuracy, per-class recall, confusion matrix`

> 🎬 **See this illustrated and interactive:** [https://pump-cavitation-audio.streamlit.app/?stage=audit](https://pump-cavitation-audio.streamlit.app/?stage=audit)

## Part 4 · Technical explanation

In [ ]:
test_probs=model.predict(X_test,verbose=0);test_pred=np.argmax(test_probs,axis=1)
print(classification_report(y_test,test_pred,target_names=CLASSES))
cm=confusion_matrix(y_test,test_pred)
ConfusionMatrixDisplay(cm,display_labels=CLASSES).plot(cmap="Blues",xticks_rotation=20)
plt.title("Unseen synthetic clips");plt.show()
print("Limitations: synthetic audio, one simulated pump family, simplified noise, no microphone/domain-shift study.")
print("A real study must use controlled labelled pump tests and independent validation across pumps, loads, microphones, and sites.")

## Part 5 · What you just built

**In the notebook:** Report accuracy, classification report, and confusion matrix on unseen clips.

**Takeaway:** Audit the errors that matter mechanically, not only the total number correct.

◀ [Previous: From Sound to Pump Condition](https://pump-cavitation-audio.streamlit.app/?stage=prediction) &nbsp;|&nbsp; [Project overview](https://pump-cavitation-audio.streamlit.app/?stage=start)

---
# Final engineering conclusion

The project connects three disciplines: cavitation physics supplies the phenomenon, signal processing makes its sound visible, and a 2D CNN learns time-frequency patterns. The final output is a condition probability and a **simulated** response—not an autonomous industrial control instruction.